# Purpose of this notebook
The purpose is to perform step 1 of the recommended project workflow to check data quality and split once, then lock the test set away

In [1]:
import numpy as np
import pandas as pd

In [2]:
wine_white = pd.read_csv("datasets/winequality-white.csv", delimiter=';')

# Data Quality Check

## Size and data types

In [3]:
wine_white.shape

(4898, 12)

In [4]:
wine_white.dtypes

fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

## Null values
No missing values

In [5]:
wine_white.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

## Duplicate values

In [6]:
wine_white.duplicated().sum()

np.int64(937)

### Same wine recorded twice, or different wines?
If duplicates are repeat records of one wine, keeping them lets the same wine sit in both a training and a validation fold (or in both train and test), which inflates scores. If they were different wines that happen to measure identically, they should be kept. We check two things: whether copies ever disagree on `quality`, and how far apart copies sit in the file.

In [7]:
dup_rows = wine_white[wine_white.duplicated(keep=False)]
features = wine_white.columns.drop('quality').tolist()

# Do copies with identical measurements ever have different quality scores?
quality_per_group = dup_rows.groupby(features)['quality'].nunique()
print('Duplicate groups:', len(quality_per_group))
print('Groups whose copies disagree on quality:', (quality_per_group > 1).sum())

# How far apart (in rows) are the copies of each group?
group_id = dup_rows.groupby(features).ngroup()
gap = dup_rows.index.to_series().groupby(group_id).agg(lambda s: s.max() - s.min())
print('Median row gap between copies:', gap.median())
print('Share of groups with copies within 5 rows:', round((gap <= 5).mean(), 2))
print('Copies per group:')
print(dup_rows.groupby(group_id).size().value_counts().sort_index())

Duplicate groups: 772
Groups whose copies disagree on quality: 0
Median row gap between copies: 4.0
Share of groups with copies within 5 rows: 0.6
Copies per group:
2    673
3     61
4     24
5      6
6      4
7      2
8      2
Name: count, dtype: int64


All 11 measurements match exactly (several to 4-5 decimal places), copies never disagree on quality, and most copies sit within a few rows of each other. We judge these to be the same wine recorded more than once, not distinct wines. Keeping one row per wine is the simplest correct treatment: it removes the leakage that grouped CV would otherwise have to handle, and doing it before the split also stops one wine appearing in both train and test.

In [8]:
wine_white = wine_white.drop_duplicates().reset_index(drop=True)

In [9]:
wine_white.shape

(3961, 12)

## Class balance
We binarise `quality` (good = 7 or above). Only the class proportion is checked here; how predictors relate to the outcome is left to EDA on the training set.

In [10]:
print('Share of good wines (quality >= 7):', (wine_white['quality'] >= 7).mean().round(3))
wine_white['quality'].value_counts().sort_index()

Share of good wines (quality >= 7): 0.208


quality
3      20
4     153
5    1175
6    1788
7     689
8     131
9       5
Name: count, dtype: int64

About 21% of wines are good, so the classes are imbalanced. Accuracy alone would be misleading. We use stratified splitting and stratified CV, and PR curves alongside ROC curves.

We identify 937 duplicate rows and drop them, leaving us with 3961 rows

## Domain specific checks

### Checking number of negative values in features (expect 0)
Physicochemical features should not have negative values

In [11]:
numeric_cols = wine_white.select_dtypes(include=['number']).columns

# Identify negative values (< 0) per column
negative_counts = (wine_white[numeric_cols] < 0).sum()
negative_counts

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

### Check physiochemical features that should not be 0

In [12]:
# Flag chemically impossible zeros
impossible_zero_cols = ['density', 'pH', 'fixed acidity', 'alcohol']
impossible_zeros = (wine_white[impossible_zero_cols] == 0.0).sum()
print("\n--- Chemically Impossible Zeros ---")
print(impossible_zeros)


--- Chemically Impossible Zeros ---
density          0
pH               0
fixed acidity    0
alcohol          0
dtype: int64


# Split Into Training and Test Set
Use a 80/20 train test split, and stratification for `quality`

In [13]:
from sklearn.model_selection import train_test_split

# Split the entire DataFrame directly (keeps X and y together)
train_df, test_df = train_test_split(
    wine_white,
    test_size=0.20,
    random_state=211,
    stratify=wine_white['quality']  # Preserves target class proportions
)

# Save the train and test DataFrames to CSV files
train_df.to_csv("datasets/winequality-white-train.csv", index=False)
test_df.to_csv("datasets/winequality-white-test.csv", index=False)